# MLIP example: Free Energy Perturbation (FEP)

**Before you begin: Note that this notebook runs a simulation that takes around 25-35 minutes on one GPU (tested on an 80GB NVIDIA H100) using the configs provided in Sections 4 and 5. This can be made faster by reducing the number of simulation steps or running on a multi-GPU instance.**

In this notebook we demonstrate running a **Free Energy Perturbation (FEP)** simulation in `mlip` using a real solvated system: a small molecule (methane) in a pre-minimized water box (`methane_minimized.pdb` + `methane_molecule_indices.npy`).

**Note that the exact model, preparation pipeline, and setup described in this notebook were used to achieve SOTA performance (MAE=0.66 kcal/mol) on the 36-molecule hydration free energy benchmark from [Moore et al., Journal American Chemical Society, 2026](https://doi.org/10.1021/jacs.5c10940), completing each molecule in under 5 hours runtime on 2×80GB H100 GPUs (under 10 GPU-hours total).**

## Background

FEP computes the free-energy difference between two "end-states" of a system by relying on non-physical, "alchemical" transitions rather than physical pathways ([Zwanzig, JCP 1954](https://doi.org/10.1063/1.1740409); for a modern review, [Mey et al., Liv. J. Comput. Mol. Sci. 2020](https://doi.org/10.33011/livecoms.2.1.18378)). Here we compute a **hydration free energy**: the free energy of transferring the solute from vacuum into water, by gradually decoupling it from the surrounding solvent.

`mlip` implements this decoupling by scaling the edges of the MLIP graph between the solute and solvent along a three-stage alchemical path ($A \to R \to B$), following [Xie et al., JCTC 2026](https://doi.org/10.1021/acs.jctc.5c02019):

- **State A (Fully Coupled)**: the solute interacts normally with the surrounding water.
- **State R (Repulsive)**: directly removing graph edges can let decoupled atoms clash with their neighbors, causing spikes in the MLIP energy. A softcore repulsive potential is introduced here, at its maximal strength, to keep the two subsystems physically separated while the edges are switched off.
- **State B (Fully Decoupled)**: the solute is entirely isolated from the water.

Each point along this path is a **lambda ($\lambda$) window**, defined by a pair of values: the **edge weight** (1.0 at A, 0.0 at B) and the **repulsion weight** (0.0 at A and B, peaking at 1.0 at R). Every window is run as an independent MD replica, optionally exchanging configurations with its neighbors via **Hamiltonian Replica Exchange (HREX)**. The free-energy difference is then recovered from the energies sampled at every window using estimators such as BAR ([Bennett, JCP 1976](https://doi.org/10.1016/0021-9991(76)90112-2)) or MBAR ([Shirts & Chodera, JCP 2006](https://doi.org/10.1063/1.2978177)), which we compute at the end of this notebook with [pymbar](https://github.com/choderalab/pymbar).


**Why only a few lambda windows?** We typically use **18 lambda windows** for our production runs (see section **3. Configure the FEP sampler** below), each sampled for hundreds of picoseconds; a run that is unfeasible to reproduce in a tutorial. Instead of inventing an arbitrary toy schedule, this notebook takes the **real production 18-window schedule** and shows a principled way to subsample it down to a handful of windows that still trace the full $A \to R \to B$ path. This keeps the notebook fast while demonstrating the actual mechanics (and diagnostics) you would use to judge whether a given lambda spacing is adequate.

**Install and logging setup**

In [ ]:
%pip install "mlip[cuda]" pymbar pandas matplotlib

# Use this instead for installation without GPU:
# %pip install mlip pymbar pandas matplotlib

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, force=True, format="%(levelname)s - %(message)s"
)

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="advanced_simulation/*",
    local_dir="",
)

In [ ]:
import jax

print(jax.devices())

## 1. Load a pre-trained MLIP model

Unlike the other tutorials which use the v2-release models, here we load a specific model that performs better for hydration free energies. We found that the dataset released with `mlip-v1` gives better water behaviour than v2, so we retrained a ViSNet model on the v1 dataset for this notebook.

In [ ]:
from mlip.models import Visnet
from mlip.models.model_io import load_model_from_zip

model_path = "advanced_simulation/visnet-trained-on-v1-dataset.zip"
force_field = load_model_from_zip(Visnet, model_path)

## 2. Load the solvated system

Unlike the other tutorials, we don't build the system from scratch here. We load a real, pre-minimized system of a methane molecule in a rhombic dodecahedron water box, with 11A padding from solute to box edge.

We load in the following files:

- `methane_minimized.pdb`: 1 methane molecule + explicit water, already classically minimized.
- `methane_molecule_indices.npy`: precomputed per-atom molecule assignment (required by the Monte Carlo barostat for NPT simulations; see the [advanced simulation tutorial](https://github.com/instadeepai/mlip/blob/main/tutorials/advanced_simulation_tutorial.ipynb) for further details).

In [ ]:
import numpy as np
from ase.io import read as ase_read

pdb_path = "advanced_simulation/methane_minimized.pdb"
molecule_indices_path = "advanced_simulation/methane_molecule_indices.npy"

atoms = ase_read(pdb_path)
molecule_indices = np.load(molecule_indices_path).tolist()

# For this system, the solute is the first molecule, with index 0.
num_solute_atoms = molecule_indices.count(0)

print(f"Number of atoms      : {len(atoms)}")
print(f"Species              : {sorted(set(atoms.get_chemical_symbols()))}")
print(f"Cell lengths/angles  : {atoms.cell.cellpar().round(3)}")
print(f"Number of molecules  : {max(molecule_indices) + 1}")
print(f"Solute atoms (mol 0) : {num_solute_atoms}")
print(f"Solute formula       : {atoms[:num_solute_atoms].get_chemical_formula()}")

### A note on the simulation cell

The cell angles printed above are not all 90&deg; &mdash; this box is a **triclinic "rhombic dodecahedron"**, a shape commonly used when solvating a single small molecule because it packs the same minimum-image separation with noticeably less water (and hence less compute) than an equivalent cubic box.

## 3. Configure the FEP sampler

[`FEPSimulationSampler`](https://instadeepai.github.io/mlip/api_reference/simulation/fep_sampler.html#mlip.simulation.fep.sampler.FEPSimulationSampler) runs one JAX-MD engine per lambda window and dispatches them across available devices. We use this sampler class for all three simulation stages (minimization, NVT equilibration, and production NPT), so the alchemical setup below is shared across the whole notebook.

- **Alchemical MLIP potential**: `use_alchemical_mlip=True` (default) predicts alchemical energies by smoothly scaling edges inside the model's message-passing equations. Setting `False` instead mixes the MLIP potential evaluated at the two physical endpoints. `True` is faster, but the alchemical path is non-linear and its shape differs between architectures, whereas `False` provides a linear path, which may be preferable in some cases.
- **Repulsive potential**: `repulsive_potential` specifies the shape of the softcore repulsive potential to use to prevent clashes at the endpoints. We use the default setting of a softcore Lennard-Jones potential, with per-element parameters taken from GAFF-2.1.
- **Alchemical atoms**: `alchemical_atom_indices` selects which atoms are gradually decoupled from the rest of the system. This will be the solute atoms.
- **Lambda schedule**: `lambda_edge_values` and `lambda_repulsion_values` are equal-length lists, one entry per window
    - `lambda_edge_values` controls the scaling of the edges between solute and water in the MLIP graph, going from `1.0` (fully coupled) to `0.0` (fully decoupled).
    - `lambda_repulsion_values` starts at `0.0`, ramps up to the maximum strength of the softcore repulsive potential in the middle of the path at `1.0`, then removes it again. 

### Reusing (and subsampling) a production lambda schedule

Below is the 18-window schedule we commonly use for production hydration free energy runs, chosen especially for good coverage towards the fully-decoupled (B) endpoint. Running all 18 windows for a meaningful number of steps is the kind of job that belongs on a GPU/TPU cluster, not in a quick tutorial. Instead, we **subsample** this real schedule down to a handful of windows that still trace the full $A \to R \to B$ path: both endpoints, the R-stage peak (where the repulsive potential is strongest), and a few points in between.

Note that we also split the selected lambda windows into **AR stage** ($A \to R$) and **RB stage** ($R \to B$) sets. This is not relevant to the simulation sampler itself, but we will use this separation in our analysis after the run.

In [ ]:
from mlip.simulation.fep.models import SoftcoreLennardJonesPotential

use_alchemical_mlip = True
repulsive_potential = SoftcoreLennardJonesPotential()

# The solute for this system is the first `num_solute_atoms` atoms.
# Alchemical edges are between the solute and all other atoms.
alchemical_atom_indices = np.arange(num_solute_atoms)

# Production 18-window schedule.
PRODUCTION_LAMBDA_EDGE = [
    1.00,
    0.86,
    0.71,
    0.57,
    0.43,
    0.29,
    0.14,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
    0.00,
]
PRODUCTION_LAMBDA_REP = [
    0.00,
    0.14,
    0.29,
    0.43,
    0.57,
    0.71,
    0.86,
    1.00,
    0.86,
    0.71,
    0.57,
    0.43,
    0.30,
    0.20,
    0.15,
    0.10,
    0.05,
    0.00,
]

# Subsample: indices 0 (A), 7 (R peak), 17 (B), and a few lambdas in between.
DEMO_WINDOW_INDICES = [0, 2, 5, 7, 9, 11, 13, 15, 17]

lambda_edge_values = [PRODUCTION_LAMBDA_EDGE[i] for i in DEMO_WINDOW_INDICES]
lambda_repulsion_values = [PRODUCTION_LAMBDA_REP[i] for i in DEMO_WINDOW_INDICES]

for i, (e, r) in enumerate(zip(lambda_edge_values, lambda_repulsion_values)):
    print(
        f"Window {i} (production index {DEMO_WINDOW_INDICES[i]:>2}): "
        f"edge={e:.2f}, repulsion={r:.2f}"
    )

# The R state is the shared endpoint between the AR (A -> R) and RB (R -> B) stages.
R_PEAK_PRODUCTION_INDEX = 7
stage_split = DEMO_WINDOW_INDICES.index(R_PEAK_PRODUCTION_INDEX)
ar_stage_indices = list(range(stage_split + 1))
rb_stage_indices = list(range(stage_split, len(DEMO_WINDOW_INDICES)))
print(f"AR stage windows: {ar_stage_indices}")
print(f"RB stage windows: {rb_stage_indices}")

shared_sampler_kwargs = {
    "use_alchemical_mlip": use_alchemical_mlip,
    "repulsive_potential": repulsive_potential,
    "alchemical_atom_indices": alchemical_atom_indices,
    "lambda_edge_values": lambda_edge_values,
    "lambda_repulsion_values": lambda_repulsion_values,
}

## 4. Prepare each lambda window: minimization + NVT equilibration

The input structure is already classically minimized, but may not be a good starting structure for all lambda windows. We find that our FEP simulations are most stable if we start by minimizing and equilibrating **every lambda window independently** under its own alchemical potential, before starting the production NPT run.

1. **Minimization** (40fs): removes bad contacts for each alchemical potential, most importantly around the R state where the repulsive potential is at its strongest.
2. **NVT equilibration** (10ps, fixed volume): lets each window equilibrate with fixed volume before the production run lets the box fluctuate (NPT). This is most important around the B state, as the solvent molecules rush to fill the cavity left by the solute.

Both stages reuse the `alchemical_atom_indices` and lambda schedule configured above, and share the same `FEPSimulationSampler` mechanics as the production run in Section 5 below — only `simulation_config` and `use_replica_exchange` differ.

In [ ]:
from mlip.simulation.configs.jax_md_config import JaxMDSimulationConfig
from mlip.simulation.enums import MDIntegrator, SimulationType
from mlip.simulation.fep.sampler import FEPSimulationSampler

jax.config.update("jax_enable_x64", False)

minimization_engine_config = JaxMDSimulationConfig(
    simulation_type=SimulationType.MINIMIZATION,
    num_steps=400,
    num_episodes=1,
    snapshot_interval=400,
    timestep_fs=0.1,  # 400 steps * 0.1fs = 40fs total
    temperature_kelvin=None,
    molecule_indices=molecule_indices,
)

minimization_sampler_config = FEPSimulationSampler.Config(
    simulation_config=minimization_engine_config,
    use_replica_exchange=False,
    num_equilibration_episodes=0,
    **shared_sampler_kwargs,
)

minimization_sampler = FEPSimulationSampler(
    atoms, force_field, minimization_sampler_config
)
minimization_sampler.run()

# Extract minimized positions for each lambda window.
minimized_atoms_per_lambda = []
for state in minimization_sampler.engine_states:
    atoms_i = atoms.copy()
    atoms_i.set_positions(np.array(state.final_positions))
    minimized_atoms_per_lambda.append(atoms_i)

In [ ]:
nvt_engine_config = JaxMDSimulationConfig(
    simulation_type=SimulationType.MD,
    md_integrator=MDIntegrator.NVT_LANGEVIN,
    num_steps=1_000,  # 1ps demo; production uses 10ps
    num_episodes=1,
    snapshot_interval=1000,
    timestep_fs=1.0,
    temperature_kelvin=298.15,
    molecule_indices=molecule_indices,
)

nvt_sampler_config = FEPSimulationSampler.Config(
    simulation_config=nvt_engine_config,
    use_replica_exchange=False,
    num_equilibration_episodes=0,
    **shared_sampler_kwargs,
)

nvt_sampler = FEPSimulationSampler(
    minimized_atoms_per_lambda, force_field, nvt_sampler_config
)
nvt_sampler.run()

# Extract equilibrated positions and velocities for each lambda window.
atoms_for_md = []
for base_atoms, state in zip(minimized_atoms_per_lambda, nvt_sampler.engine_states):
    atoms_i = base_atoms.copy()
    atoms_i.set_positions(np.array(state.final_positions))
    atoms_i.set_velocities(np.array(state.final_velocities))
    atoms_for_md.append(atoms_i)

## 5. Configure and run the production NPT simulation

Each lambda window now starts from its own minimized-and-NVT-equilibrated structure (`atoms_for_md`, prepared in Section 4 above), carrying that window's own final positions and velocities into the production run.

**Hamiltonian replica exchange (HREX)**: with `use_replica_exchange=True` (default), adjacent windows attempt to swap configurations after every episode according to a Metropolis criterion. `num_equilibration_episodes` skips these attempts for the first few episodes (20ps in production) while each window settles into the shared NPT dynamics.

The MD settings below (temperature, pressure, barostat update interval) match the config we use in production; only `num_steps`/`num_episodes`/`num_equilibration_episodes` are reduced for this demo.

In [ ]:
fep_engine_config = JaxMDSimulationConfig(
    simulation_type=SimulationType.MD,
    md_integrator=MDIntegrator.NPT_MC_LANGEVIN,
    num_steps=15_000,  # 15ps demo; production uses 270ps
    num_episodes=150,  # 0.1ps between HREX; production uses 0.5ps
    snapshot_interval=10,
    timestep_fs=1.0,
    temperature_kelvin=298.15,
    pressure_bar=1.01325,
    barostat_update_interval=25,
    molecule_indices=molecule_indices,
    edge_capacity_multiplier=1.1,  # Improves runtime (default=1.25)
)

fep_sampler_config = FEPSimulationSampler.Config(
    simulation_config=fep_engine_config,
    use_replica_exchange=True,
    num_equilibration_episodes=50,  # 5ps before starting HREX; production uses 20ps
    **shared_sampler_kwargs,
)

**NOTE: This simulation is expected to take around 20-30 minutes on one GPU (tested on an 80GB NVIDIA H100). This can be made faster by reducing the number of simulation steps or running on a multi-GPU instance.**

With a single local device, the windows are dispatched sequentially within each episode; on a machine with multiple GPUs/TPUs they are run in parallel.

In [ ]:
sampler = FEPSimulationSampler(atoms_for_md, force_field, fep_sampler_config)
sampler.run()

## 6. Inspect the outputs

In addition to the usual logged outputs of an NPT simulation, each window's [`FEPSimulationState`](https://instadeepai.github.io/mlip/api_reference/simulation/fep_sampler.html#mlip.simulation.fep.sampler.FEPSimulationState) (accessible via `sampler.engine_states[i]`) carries `per_lambda_energies`, an array with shape `(n_snapshots, n_lambdas)`: the energy of every snapshot sampled by window `i`, evaluated under *every* lambda in the schedule. This "reduced potential matrix" is exactly what free-energy estimators like BAR or MBAR need.

The sampler also exposes `sampler.replica_exchange_log`: a list of dicts containing one entry per attempted swap between adjacent windows, detailing the Metropolis acceptance probability and whether the swap was accepted, which we can convert to a dataframe. We use this dataframe to plot the acceptance rate between all adjacent pairs, and to reconstruct the replica exchange paths throughout the simulation. We would expect swaps to be accepted more frequently with more lambda windows.

In [ ]:
n_lambdas = len(lambda_edge_values)

per_lambda_energies = [
    np.array(state.per_lambda_energies) for state in sampler.engine_states
]
print(
    f"{n_lambdas} windows, each with per_lambda_energies with shape "
    f"(n_snapshots, n_lambdas) = {per_lambda_energies[0].shape}"
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_style("darkgrid")
sns.set_context("notebook")

fig, axs = plt.subplots(1, 2, figsize=(14, 4))

repex_df = pd.DataFrame(sampler.replica_exchange_log)
repex_df["acceptance_probability"] = repex_df["acceptance_probability"].astype(float)
repex_df["accept"] = repex_df["accept"].astype(bool)

acceptance_by_pair = repex_df.groupby(["lambda_i", "lambda_j"]).agg(
    mean_acceptance_probability=("acceptance_probability", "mean"),
    empirical_accept_rate=("accept", "mean"),
    n_attempts=("accept", "size"),
)

pair_labels = [f"{i}-{j}" for i, j in acceptance_by_pair.index]

axs[0].bar(pair_labels, acceptance_by_pair["empirical_accept_rate"], color="steelblue")
axs[0].set_xlabel("Lambda window pair")
axs[0].set_ylabel("Acceptance rate")
axs[0].set_title("Replica exchange acceptance between adjacent windows")
axs[0].set_ylim(0, 1)


def reconstruct_walker_paths(
    exchange_log: pd.DataFrame, n_lambdas: int
) -> dict[int, list[int]]:
    """Track which lambda window each physical walker occupies over time."""
    current_mapping = np.arange(n_lambdas)
    walker_paths = {walker_id: [walker_id] for walker_id in range(n_lambdas)}
    for _, swaps in exchange_log.sort_values("episode_idx").groupby("episode_idx"):
        for _, row in swaps.iterrows():
            if row["accept"]:
                i, j = int(row["lambda_i"]), int(row["lambda_j"])
                current_mapping[i], current_mapping[j] = (
                    current_mapping[j],
                    current_mapping[i],
                )
        for lambda_idx, walker_id in enumerate(current_mapping):
            walker_paths[walker_id].append(lambda_idx)
    return walker_paths


walker_paths = reconstruct_walker_paths(repex_df, n_lambdas)

for path in walker_paths.values():
    axs[1].plot(path, alpha=0.7, linewidth=1.5)
axs[1].set_yticks(range(n_lambdas))
axs[1].set_yticklabels([f"$\\lambda_{{{i}}}$" for i in range(n_lambdas)])
axs[1].invert_yaxis()
axs[1].grid(axis="y", linestyle="--", alpha=0.3)
axs[1].set_xlabel("Episode index")
axs[1].set_ylabel("Lambda window")
axs[1].set_title("Replica exchange paths")

fig.tight_layout()
plt.show()

## 7. Estimate the free energy with MBAR

`mlip` runs the sampling and reports `per_lambda_energies` for every window &mdash; it does not itself implement a free-energy estimator. Here we feed that data into [pymbar](https://github.com/choderalab/pymbar)'s MBAR estimator, which pools samples from a set of windows to estimate the free-energy difference between every pair of lambda values (a generalization of BAR to more than two states).

Rather than running a single MBAR estimate across all windows, we estimate the free energy difference of the **AR stage** ($A \to R$) and **RB stage** ($R \to B$) separately then sum them. We do this because the MLIP energy can be unstable for "clashing" frames (which can occur for windows in the RB stage) if the edge weight > 0 (which is true for windows in the AR stage). We also discard the snapshots recorded during `num_equilibration_episodes` (before replica exchange, and before each window has had time to relax at its own lambda) as burn-in.

The free-energy difference between state A (index 0, fully coupled) and state B (last index, fully decoupled) is the **decoupling free energy** &mdash; the (unfavorable) cost of switching off the solute's interactions with water. The **hydration free energy** is its negative: the free energy of transferring the solute from vacuum into water.

We also compute the absolute error against the experimental hydration free energy of methane **(2.0 kcal/mol)** taken from the [FreeSolv database](https://github.com/mobleylab/freesolv).

In [ ]:
from ase.units import kB, kcal, mol
from pymbar import MBAR

EXPERIMENTAL_HFE = 2.0
EV_TO_KCAL_MOL = mol / kcal  # 1 eV in kcal/mol
kT_eV = kB * fep_engine_config.temperature_kelvin
kT_kcal = kT_eV * EV_TO_KCAL_MOL

# Discard the equilibration-phase snapshots (before HREX starts) as burn-in.
steps_per_episode = fep_engine_config.num_steps // fep_engine_config.num_episodes
snapshots_per_episode = steps_per_episode // fep_engine_config.snapshot_interval
n_discard = fep_sampler_config.num_equilibration_episodes * snapshots_per_episode

trimmed_energies = [energies[n_discard:] for energies in per_lambda_energies]
n_k = np.array([len(e) for e in trimmed_energies])
print(f"\nDiscarding {n_discard} burn-in snapshots per window, {n_k} retained.")


def compute_stage_mbar(
    energies_by_window: list[np.ndarray],
    stage_window_indices: list[int],
) -> MBAR:
    """Run MBAR restricted to one stage's own windows and lambda columns."""
    # u_kn[l, n] = reduced potential (kT) of snapshot n at lambda l
    stage_energies = [
        energies_by_window[i][:, stage_window_indices] for i in stage_window_indices
    ]
    n_k_stage = np.array([len(e) for e in stage_energies])
    u_kn_stage = np.concatenate(stage_energies, axis=0).T / kT_eV
    return MBAR(u_kn_stage, n_k_stage)


# Compute free energy differences for AR and RB separately
ar_mbar = compute_stage_mbar(trimmed_energies, ar_stage_indices)
rb_mbar = compute_stage_mbar(trimmed_energies, rb_stage_indices)

ar_results = ar_mbar.compute_free_energy_differences()
rb_results = rb_mbar.compute_free_energy_differences()
ar_delta_f_kt, ar_d_delta_f_kt = ar_results["Delta_f"], ar_results["dDelta_f"]
rb_delta_f_kt, rb_d_delta_f_kt = rb_results["Delta_f"], rb_results["dDelta_f"]

delta_g_ar = ar_delta_f_kt[0, -1] * kT_kcal  # A -> R
delta_g_rb = rb_delta_f_kt[0, -1] * kT_kcal  # R -> B

print("\n--- MBAR RESULTS: ---")
print(f"AR stage (A -> R): {delta_g_ar:+.2f} kcal/mol")
print(f"RB stage (R -> B): {delta_g_rb:+.2f} kcal/mol")

# Combine AR and RB stages by summing
delta_g_decouple = delta_g_ar + delta_g_rb
delta_g_decouple_std = np.sqrt(
    (ar_d_delta_f_kt[0, -1] * kT_kcal) ** 2 + (rb_d_delta_f_kt[0, -1] * kT_kcal) ** 2
)

print(
    f"Decoupling free energy (A -> B): {delta_g_decouple:+.2f} "
    f"+/- {delta_g_decouple_std:.2f} kcal/mol"
)
print(
    f"Hydration free energy (B -> A):  {-delta_g_decouple:+.2f} "
    f"+/- {delta_g_decouple_std:.2f} kcal/mol"
)
print(
    "---------------------"
    "\nAbsolute error vs. experimental HFE (2.0 kcal/mol): "
    f"{np.abs(EXPERIMENTAL_HFE - (-delta_g_decouple)):.2f} kcal/mol"
)

### Diagnostics: free-energy profile and phase-space overlap

Two diagnostics are especially useful when judging whether a lambda schedule (production or subsampled) is adequate:

- The **cumulative free-energy profile** along the path shows where most of the free-energy change happens.
- The **MBAR overlap matrix**, computed separately per stage, measures how much phase-space overlap exists between every pair of windows within that stage. Off-diagonal values near zero for adjacent windows indicate insufficient overlap &mdash; exactly the failure mode that subsampling from 18 down to a handful of windows risks introducing, and precisely what you would fix by adding windows back in that region.

Looking at these plots collectively: a segment with a large free-energy jump in the free energy profile usually shows as having poor adjacent overlap.

In [ ]:
fig, (ax_profile, ax_ar_overlap, ax_rb_overlap) = plt.subplots(1, 3, figsize=(14, 4))

profile_kcal = np.empty(n_lambdas)
for local_i, i in enumerate(ar_stage_indices):
    profile_kcal[i] = ar_delta_f_kt[0, local_i] * kT_kcal
for local_i, i in enumerate(rb_stage_indices):
    profile_kcal[i] = delta_g_ar + rb_delta_f_kt[0, local_i] * kT_kcal

ax_profile.plot(range(n_lambdas), -profile_kcal, marker="o", color="steelblue")
ax_profile.axvline(
    stage_split, color="gray", linestyle="--", alpha=0.6, label="AR / RB boundary"
)
ax_profile.set_xlabel("Lambda window index")
ax_profile.set_ylabel("Cumulative $\\Delta G$ (kcal/mol)")
ax_profile.set_title("Free-energy profile along the alchemical path")
ax_profile.grid(True)
ax_profile.legend()

for ax, stage_mbar, stage_indices, stage_name in [
    (ax_ar_overlap, ar_mbar, ar_stage_indices, "AR"),
    (ax_rb_overlap, rb_mbar, rb_stage_indices, "RB"),
]:
    overlap_matrix = np.array(stage_mbar.compute_overlap()["matrix"])
    stage_labels = [f"$\\lambda_{{{i}}}$" for i in stage_indices]
    sns.heatmap(
        overlap_matrix,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0.0,
        vmax=overlap_matrix.max(),
        xticklabels=stage_labels,
        yticklabels=stage_labels,
        cbar=False,
        ax=ax,
    )
    ax.set_xlabel("Target state (j)")
    ax.set_ylabel("Sampled state (i)")
    ax.set_title(f"{stage_name} stage: MBAR overlap matrix")

fig.tight_layout()
plt.show()

### Diagnostic: Free energy convergence vs. runtime

A single estimate of the hydration free energy (HFE) from the full production run doesn't tell us whether that estimate has actually **converged**, or if sampling for longer would still change the answer. The standard check is to repeat the same MBAR estimate using only the first $N$ picoseconds of production data, then gradually increase $N$ and see the impact on the estimate. A curve that shows the estimate is still changing at the end of the run suggests further sampling is required.

Note that as we only run 10ps of production in this tutorial, it is unlikely for the estimate to have converged; we usually run 250ps of production in practice.

In [ ]:
frame_frequency_fs = fep_engine_config.snapshot_interval * fep_engine_config.timestep_fs
production_runtimes_ps = np.arange(1, 11)  # 1 ps to 10 ps of production data

hfe_convergence_kcal = []
for runtime_ps in production_runtimes_ps:
    n_frames = int(runtime_ps * 1000 / frame_frequency_fs)
    windowed_energies = [energies[:n_frames] for energies in trimmed_energies]
    ar_mbar_t = compute_stage_mbar(windowed_energies, ar_stage_indices)
    rb_mbar_t = compute_stage_mbar(windowed_energies, rb_stage_indices)
    delta_g_ar_t = ar_mbar_t.compute_free_energy_differences()["Delta_f"][0, -1]
    delta_g_rb_t = rb_mbar_t.compute_free_energy_differences()["Delta_f"][0, -1]
    hfe_convergence_kcal.append(-(delta_g_ar_t + delta_g_rb_t) * kT_kcal)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    production_runtimes_ps, hfe_convergence_kcal, "o-", color="steelblue", markersize=6
)
ax.axhline(
    EXPERIMENTAL_HFE,
    color="r",
    linestyle="--",
    label="Experimental value",
)
ax.axhline(
    -delta_g_decouple,
    color="#898781",
    linestyle="--",
    linewidth=1,
    label="Final prediction",
)

ax.set_xticks(np.arange(production_runtimes_ps[0], production_runtimes_ps[-1] + 1))
ax.set_xlabel("Production runtime used (ps)")
ax.set_ylabel("Predicted HFE (kcal/mol)")
ax.set_title(
    f"Methane: HFE convergence (final prediction = {-delta_g_decouple:+.2f} kcal/mol)"
)
ax.legend()
fig.tight_layout()
plt.show()

## 8. Next steps

This notebook subsampled the production 18-window schedule down to a handful of windows and ran only a few picoseconds per window at each stage to demonstrate the `mlip` FEP interface end-to-end quickly on a real input system. For a production-scale hydration free energy calculation, expect to scale back up to the production setup described throughout this notebook:

- **All 18 (or more) lambda windows** especially through the R stage &mdash; aim for high (&gt;30%) replica exchange acceptance and strong MBAR overlap between every neighboring pair, using the diagnostics in sections 6&ndash;7 above.
- **The full per-lambda NVT equilibration** (10ps, not the 1ps used above) in Section 4, before starting production.
- **Much longer sampling per window** (hundreds of picoseconds) in Section 5, with the full 20 ps equilibration / 250 ps production split used in production.
- **Checkpointing** (`checkpoint_dir` / `checkpoint_interval_episodes` on [`FEPSimulationSamplerConfig`](https://instadeepai.github.io/mlip/api_reference/simulation/fep_sampler_config.html#mlip.simulation.fep.sampler.FEPSimulationSamplerConfig)) to make long runs resumable, and TPU/multi-GPU dispatch to run all lambda windows truly in parallel.

See the [FEP user guide](https://instadeepai.github.io/mlip/user_guide/enhanced_sampling.html#fep-simulations) for the full interface, including the two ways of computing the alchemical MLIP potential (`use_alchemical_mlip`) and the available softcore repulsive potentials.